# 1KG Wildtype

Characterise naturally occuring variants in the 1KG dataset.

In [14]:
# Automatically reload code in notebook
%load_ext autoreload
%autoreload 2

import pandas as pd
pd.set_option("display.max_columns", None)
import os
os.chdir("/grid/koo/home/schilder/projects/GenomeEncoder/data")

import sys
sys.path.append("code")
from src.utils import *
from src.pyensembl import *
from src.variant_annotation import *
from src.onekg import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Characterise CDS variants

In [7]:
mane = get_mane_transcripts()
db = get_db()
cv = get_clinvar_db()
vcf_files = list_vcf()

transcript_ids = mane.TranscriptId.tolist()[:10]

19284 MANE transcripts found.
23 VCF files found.


In [28]:
rec_anno.info.items()

[('AC', (1,)),
 ('AF', (0.000199681002413854,)),
 ('AN', 5008),
 ('NS', 2504),
 ('DP', 18376),
 ('EAS_AF', (0.0,)),
 ('AMR_AF', (0.0,)),
 ('AFR_AF', (0.0007999999797903001,)),
 ('EUR_AF', (0.0,)),
 ('SAS_AF', (0.0,)),
 ('AA', 'A|||'),
 ('CSQ',
  ('C|ENSG00000100652|ENST00000216540|Transcript|intron_variant||||||||-1||||||',)),
 ('GENCODE', ('ENST00000216540',)),
 ('FUNSEQ', (0.002899999963119626,))]

In [17]:
import pysam
from tqdm.auto import tqdm
f = vcf_files[0]
chrom = os.path.basename(f).split('.')[1]
vcf_in = pysam.VariantFile(f)
vcf_anno = get_annotation_vcf(chrom)
samples = list(vcf_in.header.samples)
verbose = 1
max_transcripts  = True
 
transcripts = get_protein_coding_transcripts(db, 
                                             verbose=verbose)
# Get relevant transcript coordinates
chr_transcripts = filter_transcripts(
    transcripts, 
    contig=chrom,
    biotype=['protein_coding'],
    id=transcript_ids,
    max_transcripts=max_transcripts,
    verbose=verbose > 1
)
# Step 1: Get all ClinVar variants in the transcripts
recs = {}
recs_anno = {}
for transcript_id in tqdm(transcript_ids):
    tx = db.transcript_by_id(transcript_id)
    ranges  = tx.coding_sequence_position_ranges
    recs[transcript_id] = []
    recs_anno[transcript_id] = []
    for range in ranges: 
        for rec in vcf_in.fetch(chrom, range[0], range[1]): 
            for rec_anno in query_annotation_vcf(vcf_anno, rec):  
                if rec.alleles[1] == rec_anno.alleles[1] and rec.start == rec_anno.start and rec.stop == rec_anno.stop:
                    recs[transcript_id].append(rec)  
                    recs_anno[transcript_id].append(rec_anno)


Retrieving transcripts.:   0%|          | 0/111163 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [84]:
# Step 2: Count mutation types in each sample
mutation_counts = {}
for transcript_id in tqdm(transcript_ids):
    mutation_counts[transcript_id] = {}
    for i, rec in enumerate(recs[transcript_id]):
        rec_anno = recs_anno[transcript_id][i]
        info = dict(rec_anno.info)
        empty_dict = {k: () for k in dict(info).keys()}
        for sample in tqdm(list(rec.samples), desc=f"Processing {transcript_id}",leave=False):
            mutation_counts[transcript_id][sample] = empty_dict
            sample_data = rec.samples[sample]
            gt = sample_data.allele_indices
            if gt[0] == 0 and gt[1] == 0:
                continue
            if rec_anno.start == rec.start and rec_anno.stop == rec.stop and rec_anno.ref == rec.ref:
                for allele in [rec.alleles[gt[0]], rec.alleles[gt[1]]]:
                    if allele == rec_anno.alleles[1]:
                        # for each key in info, add the value to the corresponding key in mutation_counts[transcript_id][sample]
                        for k, v in info.items():
                            mutation_counts[transcript_id][sample][k] += v




  0%|          | 0/10 [00:00<?, ?it/s]

Processing ENST00000263100:   0%|          | 0/2548 [00:00<?, ?it/s]

Processing ENST00000393078:   0%|          | 0/2548 [00:00<?, ?it/s]

Processing ENST00000393078:   0%|          | 0/2548 [00:00<?, ?it/s]

Processing ENST00000393078:   0%|          | 0/2548 [00:00<?, ?it/s]

...

Processing ENST00000261772:   0%|          | 0/2548 [00:00<?, ?it/s]

Processing ENST00000261772:   0%|          | 0/2548 [00:00<?, ?it/s]

Processing ENST00000261772:   0%|          | 0/2548 [00:00<?, ?it/s]

Processing ENST00000261772:   0%|          | 0/2548 [00:00<?, ?it/s]

Processing ENST00000261772:   0%|          | 0/2548 [00:00<?, ?it/s]

In [30]:
dict(rec_anno.info)

{'AC': (1,),
 'AF': (0.000199681002413854,),
 'AN': 5008,
 'NS': 2504,
 'DP': 18376,
 'EAS_AF': (0.0,),
 'AMR_AF': (0.0,),
 'AFR_AF': (0.0007999999797903001,),
 'EUR_AF': (0.0,),
 'SAS_AF': (0.0,),
 'AA': 'A|||',
 'CSQ': ('C|ENSG00000100652|ENST00000216540|Transcript|intron_variant||||||||-1||||||',),
 'GENCODE': ('ENST00000216540',),
 'FUNSEQ': (0.002899999963119626,)}